This template shows how to use Plotly.NET to create charts, since charting is a common feature in research.

## Preparation

Load assemblies, import the QuantConnect, Plotly.NET, and Accord packages, then get some historical data to plot.

In [ ]:
// Load the assembly files and data types in their own cell.
#load "../Initialize.csx"

In [ ]:
// Load the necessary assembly files.
#load "../QuantConnect.csx"
#r "nuget: Plotly.NET"
#r "nuget: Plotly.NET.Interactive"

In [ ]:
// Import the QuantConnect, Plotly.NET, and Accord packages for calculation and plotting.
using QuantConnect;
using QuantConnect.Research;

using Plotly.NET;
using Plotly.NET.Interactive;
using Plotly.NET.LayoutObjects;

using Accord.Math;
using Accord.Statistics;

In [ ]:
// Get historical data for a bank sector ETF and some banking companies over 2021.
var qb = new QuantBook();
var tickers = new[]
{
    "XLF",  // Financial Select Sector SPDR Fund
    "COF",  // Capital One Financial Corporation
    "GS",   // Goldman Sachs Group, Inc.
    "JPM",  // J P Morgan Chase & Co
    "WFC"   // Wells Fargo & Company
};
var symbols = tickers.Select(ticker => qb.AddEquity(ticker, Resolution.Daily).Symbol).ToList();
var history = qb.History(symbols, new DateTime(2021, 1, 1), new DateTime(2022, 1, 1)).ToList();

## Candlestick Chart

In [ ]:
// Select a symbol to plot its candlestick plot.
var symbol = symbols.First();

// Call the Chart2D.Chart.Candlestick constructor with the time and OHLC price IEnumerable to create the candlestick plot.
var bars = history.Select(slice => slice.Bars[symbol]);
var candlestick = Chart2D.Chart.Candlestick<decimal, decimal, decimal, decimal, DateTime, string>(
    bars.Select(x => x.Open),
    bars.Select(x => x.High),
    bars.Select(x => x.Low),
    bars.Select(x => x.Close),
    bars.Select(x => x.EndTime)
);

// Create the layout.
LinearAxis candleX = new LinearAxis();
candleX.SetValue("title", "Time");
LinearAxis candleY = new LinearAxis();
candleY.SetValue("title", "Price ($)");
Layout candleLayout = new Layout();
candleLayout.SetValue("xaxis", candleX);
candleLayout.SetValue("yaxis", candleY);
candleLayout.SetValue("title", Title.init($"{symbol} OHLC"));

// Assign the Layout and show the plot.
candlestick.WithLayout(candleLayout);
display(candlestick);

## Line Chart

In [ ]:
// Plot the volume of the first symbol as a line chart.
var lineBars = history.Select(slice => slice.Bars[symbol]);
var line = Chart2D.Chart.Line<DateTime, decimal, string>(
    lineBars.Select(x => x.EndTime),
    lineBars.Select(x => x.Volume)
);

LinearAxis lineX = new LinearAxis();
lineX.SetValue("title", "Time");
LinearAxis lineY = new LinearAxis();
lineY.SetValue("title", "Volume");
Layout lineLayout = new Layout();
lineLayout.SetValue("xaxis", lineX);
lineLayout.SetValue("yaxis", lineY);
lineLayout.SetValue("title", Title.init($"{symbol} Volume"));

line.WithLayout(lineLayout);
display(line);

## Scatter Plot

In [ ]:
// Plot the closing-price relationship between two symbols as a scatter plot.
var symbol1 = symbols.First();
var symbol2 = symbols.Last();

var scatter = Chart2D.Chart.Point<decimal, decimal, string>(
    history.Select(slice => slice.Bars[symbol1].Close),
    history.Select(slice => slice.Bars[symbol2].Close)
);

LinearAxis scatterX = new LinearAxis();
scatterX.SetValue("title", $"{symbol1} Price ($)");
LinearAxis scatterY = new LinearAxis();
scatterY.SetValue("title", $"{symbol2} Price ($)");
Layout scatterLayout = new Layout();
scatterLayout.SetValue("xaxis", scatterX);
scatterLayout.SetValue("yaxis", scatterY);
scatterLayout.SetValue("title", Title.init($"{symbol1} vs {symbol2}"));

scatter.WithLayout(scatterLayout);
display(scatter);

## Heat Map

In [ ]:
// Compute the daily returns of each stock.
var data = history.SelectMany(x => x.Bars.Values)
    .GroupBy(x => x.Symbol)
    .Select(x =>
    {
        var prices = x.Select(b => (double)b.Close).ToArray();
        return Enumerable.Range(0, prices.Length - 1)
            .Select(i => prices[i + 1] / prices[i] - 1).ToArray();
    }).ToArray().Transpose();

// Build the correlation matrix.
var corrMatrix = Measures.Correlation(data).Select(x => x.ToList()).ToList();

// Plot the heat map.
var X = Enumerable.Range(0, tickers.Length).ToList();
var heatmap = Plotly.NET.Chart2D.Chart.Heatmap<IEnumerable<double>, double, int, int, string>(
    zData: corrMatrix,
    X: X,
    Y: X,
    ShowScale: true,
    ReverseYAxis: true
);

var heatAxis = new LinearAxis();
heatAxis.SetValue("tickvals", X);
heatAxis.SetValue("ticktext", tickers);
var heatLayout = new Layout();
heatLayout.SetValue("xaxis", heatAxis);
heatLayout.SetValue("yaxis", heatAxis);
heatLayout.SetValue("title", Title.init("Banking Stocks and bank sector ETF Correlation Heat Map"));

heatmap.WithLayout(heatLayout);
display(heatmap);

## 3D Chart

In [ ]:
// Plot the closing-price correlation between three symbols in 3D.
var s1 = symbols[0];
var s2 = symbols[1];
var s3 = symbols[2];

var chart3d = Chart3D.Chart.Point3D<decimal, decimal, decimal, string>(
    history.Select(slice => slice.Bars[s1].Close),
    history.Select(slice => slice.Bars[s2].Close),
    history.Select(slice => slice.Bars[s3].Close)
);

LinearAxis xAxis3d = new LinearAxis();
xAxis3d.SetValue("title", $"{s1} Price ($)");
LinearAxis yAxis3d = new LinearAxis();
yAxis3d.SetValue("title", $"{s2} Price ($)");
LinearAxis zAxis3d = new LinearAxis();
zAxis3d.SetValue("title", $"{s3} Price ($)");
Layout layout3d = new Layout();
layout3d.SetValue("xaxis", xAxis3d);
layout3d.SetValue("yaxis", yAxis3d);
layout3d.SetValue("zaxis", zAxis3d);
layout3d.SetValue("title", Title.init($"{s1} vs {s2} vs {s3}"));

chart3d.WithLayout(layout3d);
display(chart3d);